In [1]:
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans
from dataclasses import dataclass, field

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _get(d: Dict[str, Any], k: str, default=None):
    return d.get(k, default)


@dataclass
class NERMetricsCollector:
    overall_rows: List[Dict[str, Any]] = field(default_factory=list)
    label_rows: List[Dict[str, Any]] = field(default_factory=list)
    meta: Dict[str, Any] = field(
        default_factory=dict
    )  # ex.: nome do modelo, dataset, etc.

    def record(
        self,
        split_name: Any,
        metrics: Dict[str, Any],
        extras: Optional[Dict[str, Any]] = None,
    ):
        """
        Registra os resultados de um split.
        - split_name: pode ser string, int, tupla... será convertido para string
        - metrics: dict retornado pelo seu train/eval
        - extras: (opcional) dict com metadados (seed, versão, etc.)
        """
        split_str = str(split_name)

        # ------------------------------
        # Tabela 1: métricas gerais
        # ------------------------------
        overall = {
            "split": split_str,
            "eval_loss": _get(metrics, "eval_loss"),
            "overall_precision": _get(metrics, "eval_overall_precision"),
            "overall_recall": _get(metrics, "eval_overall_recall"),
            "overall_f1": _get(metrics, "eval_overall_f1"),
            "overall_accuracy": _get(metrics, "eval_overall_accuracy"),
            "f1_micro": _get(metrics, "eval_f1_micro"),
            "f1_macro": _get(metrics, "eval_f1_macro"),
            "f1_weighted": _get(metrics, "eval_f1_weighted"),
            "runtime_s": _get(metrics, "eval_runtime"),
            "samples_per_sec": _get(metrics, "eval_samples_per_second"),
            "steps_per_sec": _get(metrics, "eval_steps_per_second"),
            "epoch": _get(metrics, "epoch"),
        }

        self.overall_rows.append(overall)

        # ------------------------------
        # Tabela 2: métricas por rótulo
        # ------------------------------
        # Regra: qualquer entrada do dict que seja outro dict contendo
        # precision/recall/f1/number é tratada como rótulo.
        for k, v in metrics.items():
            if isinstance(v, dict) and {"precision", "recall", "f1", "number"} <= set(
                v.keys()
            ):
                self.label_rows.append(
                    {
                        "split": split_str,
                        "label": k.replace(
                            "eval_", ""
                        ),  # remove prefixo "eval_" para ficar limpo
                        "precision": v["precision"],
                        "recall": v["recall"],
                        "f1": v["f1"],
                        "support": v["number"],
                    }
                )

    # Comentário: retorna DataFrames prontos para inspeção ou export
    def to_dataframes(self):
        df_overall = pd.DataFrame(self.overall_rows)
        df_labels = pd.DataFrame(self.label_rows)
        return df_overall, df_labels

    # Comentário: exporta dois CSVs (UTF-8 com BOM para abrir liso no Excel)
    def to_csv(self, base_name: str = "ner"):
        df_overall, df_labels = self.to_dataframes()
        df_overall.to_csv(
            f"{base_name}_overall_metrics.csv", index=False, encoding="utf-8-sig"
        )
        df_labels.to_csv(
            f"{base_name}_label_metrics.csv", index=False, encoding="utf-8-sig"
        )
        return f"{base_name}_overall_metrics.csv", f"{base_name}_label_metrics.csv"


# --- Cria (ou reaproveita) um coletor global ---
if "ner_collector" not in globals():
    ner_collector = NERMetricsCollector()

# Configuração e Verificação Inicial

In [3]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

MODEL_NAME = "xlm-roberta-base"

In [4]:
lener_ds = load_dataset("peluz/lener_br", trust_remote_code = True)

In [5]:
lener_ds

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1390
    })
})

In [6]:
lener_full = concatenate_datasets(
    [lener_ds["train"], lener_ds["validation"], lener_ds["test"]]
)

In [7]:
tags = [
    "O",
    "B-ORGANIZACAO",
    "I-ORGANIZACAO",
    "B-PESSOA",
    "I-PESSOA",
    "B-TEMPO",
    "I-TEMPO",
    "B-LOCAL",
    "I-LOCAL",
    "B-LEGISLACAO",
    "I-LEGISLACAO",
    "B-JURISPRUDENCIA",
    "I-JURISPRUDENCIA",
]

In [8]:
label2id = {l: i for i, l in enumerate(tags)}
id2label = {i: l for i, l in enumerate(tags)}
NUM_LABELS = len(tags)

In [9]:
label2id

{'O': 0,
 'B-ORGANIZACAO': 1,
 'I-ORGANIZACAO': 2,
 'B-PESSOA': 3,
 'I-PESSOA': 4,
 'B-TEMPO': 5,
 'I-TEMPO': 6,
 'B-LOCAL': 7,
 'I-LOCAL': 8,
 'B-LEGISLACAO': 9,
 'I-LEGISLACAO': 10,
 'B-JURISPRUDENCIA': 11,
 'I-JURISPRUDENCIA': 12}

In [10]:
def decode_labels(example):
    example["ner_tags_str"] = [tags[i] for i in example["ner_tags"]]
    return example


lener_full = lener_full.map(decode_labels)

In [11]:
lener_full

Dataset({
    features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
    num_rows: 10395
})

In [12]:
NUM_LABELS

13

# Splits

In [13]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [14]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags_str"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags_str"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [16]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    print(f"Selecionando {int(pct_test*len(dataset))} sentenças para teste...")
    while len(test_idx) < int(pct_test*len(dataset)):
        print(f"  {len(test_idx)} selecionadas...")
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [17]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [20]:
def standard_split_conll(dataset, 
                         pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42):

    ds = DatasetDict(
        {
            ("val" if k == "dev" or k == "validation" else k): v
            for k, v in lener_ds.items()
        }
    )
    return ds

In [21]:
standard_split = standard_split_conll(lener_full)
# print('std')
# # random_splt = random_splits(lener_full)
# # print('random')
# heur_len = heur_len_split(lener_full)
# print("heur_len")
# heur_rare = heur_rare_split(lener_full)
# print("heur_rare")
# advers = adversarial_split(lener_full)
# print("advs")
# loc = loc_split(lener_full)
# print("loc")
# semantic = semantic_cluster_split(lener_full)
# print("semantic")
# reverse = reverse_curriculum_split(lener_full)
# print("reverse")

In [22]:
standard_split

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 7828
    })
    val: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1390
    })
})

# Experimentos

In [23]:
from sklearn.metrics import f1_score as skl_f1

In [24]:
id2label

{0: 'O',
 1: 'B-ORGANIZACAO',
 2: 'I-ORGANIZACAO',
 3: 'B-PESSOA',
 4: 'I-PESSOA',
 5: 'B-TEMPO',
 6: 'I-TEMPO',
 7: 'B-LOCAL',
 8: 'I-LOCAL',
 9: 'B-LEGISLACAO',
 10: 'I-LEGISLACAO',
 11: 'B-JURISPRUDENCIA',
 12: 'I-JURISPRUDENCIA'}

In [25]:
id2label[0]

'O'

In [26]:
label2id

{'O': 0,
 'B-ORGANIZACAO': 1,
 'I-ORGANIZACAO': 2,
 'B-PESSOA': 3,
 'I-PESSOA': 4,
 'B-TEMPO': 5,
 'I-TEMPO': 6,
 'B-LOCAL': 7,
 'I-LOCAL': 8,
 'B-LEGISLACAO': 9,
 'I-LEGISLACAO': 10,
 'B-JURISPRUDENCIA': 11,
 'I-JURISPRUDENCIA': 12}

In [27]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc": loc_split,
            "reverse": reverse_curriculum_split,
            "semantic": semantic_cluster_split,
            "heur_len": heur_len_split,
            "heur_rare": heur_rare_split,
            "std": standard_split_conll,
            "advs": adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = list(ds["train"].features["ner_tags"].feature.names)
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        # usa 'ner_tags_str' se existir; senão, usa 'ner_tags' (ids)
        labels_field = "ner_tags_str" if "ner_tags_str" in examples else "ner_tags"

        def _to_id(x):  # converte string->id ou id->id
            return label2id[x] if isinstance(x, str) else int(x)

        labels_batch = []
        for i, word_labels in enumerate(examples[labels_field]):
            word_ids = tokenized.word_ids(batch_index=i)
            label_ids = []
            prev = None
            for w in word_ids:
                if w is None:
                    label_ids.append(-100)
                elif w != prev:
                    label_ids.append(_to_id(word_labels[w]))
                else:
                    label_ids.append(_to_id(word_labels[w]) if label_all_tokens else -100)
                prev = w
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []  # p/ seqeval
        flat_preds, flat_labels = [], []  # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)
        
        #sent_preds_str = [str(s) for sentence in sent_preds]
        #sent_labels_str = [str(s) for sentence in sent_labels]

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro = skl_f1(flat_labels, flat_preds, average="micro", zero_division=0)
        f1_macro = skl_f1(flat_labels, flat_preds, average="macro", zero_division=0)
        f1_weighted = skl_f1(
            flat_labels, flat_preds, average="weighted", zero_division=0
        )

        return {
            **seqeval_metrics,  # overall_precision / recall / f1
            "f1_micro": f1_micro,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }

    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        # output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="no",
        # load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none",
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [28]:
# -- helpers -------------------------------------------------------------
def _label_names_from_schema(ds: Dataset) -> list[str]:
    feat = ds.features["ner_tags"]
    if isinstance(feat, Sequence) and isinstance(feat.feature, ClassLabel):
        return list(feat.feature.names)
    # fallback: cria nomes artificiais (ideal é fazer cast p/ ClassLabel antes)
    max_id = int(max(max(row) for row in ds["ner_tags"])) if len(ds) else -1
    return [f"L{i}" for i in range(max_id + 1)]

def ensure_ner_strings(ds: Dataset) -> Dataset:
    names = _label_names_from_schema(ds)
    def _map_batch(batch):
        return {"ner_tags_str": [[names[i] for i in seq] for seq in batch["ner_tags"]]}
    return ds.map(_map_batch, batched=True)

def sanitize(ds: Dataset) -> Dataset:
    # filtra linhas vazias/inconsistentes
    def _ok(ex):
        toks = ex.get("tokens", None)
        labs = ex.get("ner_tags", None)
        return (
            isinstance(toks, list) and isinstance(labs, list) and
            len(toks) > 0 and len(labs) > 0 and len(toks) == len(labs)
        )
    return ds.filter(_ok)

# -- sanear qualquer tipo ------------------------------------------------
def sanitize_any(ds_or_dd):
    if isinstance(ds_or_dd, DatasetDict):
        out = {}
        for k, ds in ds_or_dd.items():
            ds = sanitize(ds)
            if "ner_tags_str" not in ds.column_names:
                ds = ensure_ner_strings(ds)
            out[k] = ds
        return DatasetDict(out)
    elif isinstance(ds_or_dd, Dataset):
        ds = sanitize(ds_or_dd)
        if "ner_tags_str" not in ds.column_names:
            ds = ensure_ner_strings(ds)
        return ds
    else:
        raise TypeError(f"Esperado Dataset ou DatasetDict, recebi: {type(ds_or_dd)}")

In [29]:
lener_full = sanitize_any(lener_full)


In [30]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "std", "advs"]

In [31]:
results = {}
trainer_all = {}
s = splits[0]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer

print("F1 Macro:", metrics["eval_f1_macro"])

print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: loc


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([13]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([13, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2078/2078 [00:00<00:00, 16769.15 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_15840\2617588520.py:175: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Jurisprudencia,Legislacao,Local,Organizacao,Pessoa,Tempo,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.050401,"{'precision': 0.7586206896551724, 'recall': 0.6875, 'f1': 0.7213114754098361, 'number': 32}","{'precision': 0.4888888888888889, 'recall': 0.6666666666666666, 'f1': 0.5641025641025641, 'number': 33}","{'precision': 1.0, 'recall': 0.8571428571428571, 'f1': 0.923076923076923, 'number': 7}","{'precision': 0.7971014492753623, 'recall': 0.8333333333333334, 'f1': 0.8148148148148148, 'number': 66}","{'precision': 0.9117647058823529, 'recall': 0.9117647058823529, 'f1': 0.9117647058823528, 'number': 34}","{'precision': 1.0, 'recall': 0.9375, 'f1': 0.967741935483871, 'number': 32}",0.779343,0.813725,0.796163,0.985020,0.985020,0.867778,0.985351
2,0.123700,0.031704,"{'precision': 0.6756756756756757, 'recall': 0.78125, 'f1': 0.7246376811594203, 'number': 32}","{'precision': 0.7419354838709677, 'recall': 0.696969696969697, 'f1': 0.71875, 'number': 33}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 7}","{'precision': 0.8208955223880597, 'recall': 0.8333333333333334, 'f1': 0.8270676691729324, 'number': 66}","{'precision': 0.8857142857142857, 'recall': 0.9117647058823529, 'f1': 0.8985507246376812, 'number': 34}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 32}",0.827751,0.848039,0.837772,0.992609,0.992609,0.936886,0.992526
3,0.028600,0.042217,"{'precision': 0.5833333333333334, 'recall': 0.65625, 'f1': 0.6176470588235293, 'number': 32}","{'precision': 0.7575757575757576, 'recall': 0.7575757575757576, 'f1': 0.7575757575757576, 'number': 33}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 7}","{'precision': 0.8285714285714286, 'recall': 0.8787878787878788, 'f1': 0.8529411764705883, 'number': 66}","{'precision': 0.8857142857142857, 'recall': 0.9117647058823529, 'f1': 0.8985507246376812, 'number': 34}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 32}",0.816901,0.852941,0.834532,0.989554,0.989554,0.922940,0.989010
4,0.016600,0.034770,"{'precision': 0.5365853658536586, 'recall': 0.6875, 'f1': 0.6027397260273972, 'number': 32}","{'precision': 0.6486486486486487, 'recall': 0.7272727272727273, 'f1': 0.6857142857142857, 'number': 33}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 7}","{'precision': 0.7945205479452054, 'recall': 0.8787878787878788, 'f1': 0.8345323741007193, 'number': 66}","{'precision': 0.8857142857142857, 'recall': 0.9117647058823529, 'f1': 0.8985507246376812, 'number': 34}","{'precision': 0.9696969696969697, 'recall': 1.0, 'f1': 0.9846153846153847, 'number': 32}",0.769912,0.852941,0.809302,0.990342,0.990342,0.892607,0.989970
5,0.008400,0.034374,"{'precision': 0.5526315789473685, 'recall': 0.65625, 'f1': 0.6, 'number': 32}","{'precision': 0.65, 'recall': 0.7878787878787878, 'f1': 0.7123287671232875, 'number': 33}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 7}","{'precision': 0.8405797101449275, 'recall': 0.8787878787878788, 'f1': 0.8592592592592593, 'number': 66}","{'precision': 0.8857142857142857, 'recall': 0.9117647058823529, 'f1': 0.8985507246376812, 'number': 34}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 32}",0.791855,0.857843,0.823529,0.991722,0.991722,0.932437,0.991450


F1 Macro: 0.8749080418719243
F1 Weighted: 0.9877434316251948
{'eval_loss': 0.07158570736646652, 'eval_JURISPRUDENCIA': {'precision': 0.75, 'recall': 0.8225806451612904, 'f1': 0.7846153846153845, 'number': 62}, 'eval_LEGISLACAO': {'precision': 0.75, 'recall': 0.78, 'f1': 0.7647058823529411, 'number': 50}, 'eval_LOCAL': {'precision': 0.8813559322033898, 'recall': 0.7761194029850746, 'f1': 0.8253968253968255, 'number': 67}, 'eval_ORGANIZACAO': {'precision': 0.8230088495575221, 'recall': 0.8532110091743119, 'f1': 0.8378378378378378, 'number': 109}, 'eval_PESSOA': {'precision': 0.6236559139784946, 'recall': 0.7073170731707317, 'f1': 0.6628571428571428, 'number': 82}, 'eval_TEMPO': {'precision': 0.8928571428571429, 'recall': 0.9375, 'f1': 0.9146341463414636, 'number': 80}, 'eval_overall_precision': 0.7846481876332623, 'eval_overall_recall': 0.8177777777777778, 'eval_overall_f1': 0.8008705114254624, 'eval_overall_accuracy': 0.9879003192256204, 'eval_f1_micro': 0.9879003192256204, 'eval_f1_mac

20

In [32]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [33]:
!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [34]:
import time

In [35]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [36]:
results = {}
trainer_all = {}
s = splits[1]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: reverse


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([13]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([13, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2078/2078 [00:00<00:00, 4526.07 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_15840\2617588520.py:175: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Jurisprudencia,Legislacao,Local,Organizacao,Pessoa,Tempo,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.203763,"{'precision': 0.5740072202166066, 'recall': 0.46355685131195334, 'f1': 0.5129032258064516, 'number': 343}","{'precision': 0.752755905511811, 'recall': 0.7823240589198036, 'f1': 0.767255216693419, 'number': 611}","{'precision': 1.0, 'recall': 0.05555555555555555, 'f1': 0.10526315789473684, 'number': 36}","{'precision': 0.7210365853658537, 'recall': 0.7276923076923076, 'f1': 0.7243491577335376, 'number': 650}","{'precision': 0.8490028490028491, 'recall': 0.8186813186813187, 'f1': 0.8335664335664336, 'number': 364}","{'precision': 0.9291338582677166, 'recall': 0.8838951310861424, 'f1': 0.9059500959692899, 'number': 267}",0.756782,0.724791,0.740441,0.956665,0.956665,0.727468,0.953016
2,0.063500,0.153870,"{'precision': 0.5575221238938053, 'recall': 0.5510204081632653, 'f1': 0.5542521994134897, 'number': 343}","{'precision': 0.7298050139275766, 'recall': 0.8576104746317512, 'f1': 0.7885628291948835, 'number': 611}","{'precision': 0.8461538461538461, 'recall': 0.3055555555555556, 'f1': 0.44897959183673475, 'number': 36}","{'precision': 0.7605042016806722, 'recall': 0.8353846153846154, 'f1': 0.7961876832844575, 'number': 650}","{'precision': 0.8715846994535519, 'recall': 0.8763736263736264, 'f1': 0.873972602739726, 'number': 364}","{'precision': 0.9202898550724637, 'recall': 0.951310861423221, 'f1': 0.9355432780847146, 'number': 267}",0.758450,0.810216,0.783479,0.968679,0.968679,0.817392,0.966779
3,0.011400,0.193725,"{'precision': 0.6528662420382165, 'recall': 0.597667638483965, 'f1': 0.624048706240487, 'number': 343}","{'precision': 0.7027439024390244, 'recall': 0.7545008183306056, 'f1': 0.7277032359905288, 'number': 611}","{'precision': 0.625, 'recall': 0.2777777777777778, 'f1': 0.3846153846153846, 'number': 36}","{'precision': 0.7722222222222223, 'recall': 0.8553846153846154, 'f1': 0.8116788321167883, 'number': 650}","{'precision': 0.8974358974358975, 'recall': 0.8653846153846154, 'f1': 0.8811188811188811, 'number': 364}","{'precision': 0.9318181818181818, 'recall': 0.9213483146067416, 'f1': 0.9265536723163842, 'number': 267}",0.772512,0.789520,0.780923,0.965904,0.965904,0.824589,0.964115
4,0.004000,0.184396,"{'precision': 0.6621160409556314, 'recall': 0.565597667638484, 'f1': 0.610062893081761, 'number': 343}","{'precision': 0.8018154311649016, 'recall': 0.867430441898527, 'f1': 0.8333333333333333, 'number': 611}","{'precision': 0.6, 'recall': 0.4166666666666667, 'f1': 0.49180327868852464, 'number': 36}","{'precision': 0.7827338129496403, 'recall': 0.8369230769230769, 'f1': 0.8089219330855018, 'number': 650}","{'precision': 0.8338557993730408, 'recall': 0.7307692307692307, 'f1': 0.7789165446559297, 'number': 364}","{'precision': 0.9391634980988594, 'recall': 0.9250936329588015, 'f1': 0.9320754716981132, 'number': 267}",0.796099,0.790841,0.793461,0.967892,0.967892,0.833520,0.966161
5,0.001900,0.184323,"{'precision': 0.6655629139072847, 'recall': 0.5860058309037901, 'f1': 0.6232558139534884, 'number': 343}","{'precision': 0.8065015479876161, 'recall': 0.8527004909983633, 'f1': 0.8289578361177407, 'number': 611}","{'precision': 0.6086956521739131, 'recall': 0.3888888888888889, 'f1': 0.47457627118644075, 'number': 36}","{'precision': 0.7833089311859444, 'recall': 0.823076923076923, 'f1': 0.8027006751687921, 'number': 650}","{'precision': 0.884393063583815, 'recall': 0.8406593406593407, 'f1': 0.8619718309859156, 'number': 364}","{'precision': 0.9328358208955224, 'recall': 0.9363295880149812, 'f1': 0.9345794392523363, 'number': 267}",0.805556,0.804491,0.805023,0.969052,0.969052,0.839635,0.967443


F1 Macro: 0.7213917323281355
F1 Micro: 0.9058980526821307
F1 Weighted: 0.8970099617174927
{'eval_loss': 0.6189872026443481, 'eval_JURISPRUDENCIA': {'precision': 0.41245972073039744, 'recall': 0.3991683991683992, 'f1': 0.4057052297939778, 'number': 962}, 'eval_LEGISLACAO': {'precision': 0.5343415248897291, 'recall': 0.6352059925093633, 'f1': 0.5804243668720055, 'number': 1335}, 'eval_LOCAL': {'precision': 0.7040816326530612, 'recall': 0.2923728813559322, 'f1': 0.41317365269461076, 'number': 708}, 'eval_ORGANIZACAO': {'precision': 0.6198830409356725, 'recall': 0.7268571428571429, 'f1': 0.6691215149921095, 'number': 1750}, 'eval_PESSOA': {'precision': 0.7979066022544283, 'recall': 0.6615487316421896, 'f1': 0.7233576642335765, 'number': 1498}, 'eval_TEMPO': {'precision': 0.9250788643533123, 'recall': 0.8806306306306306, 'f1': 0.9023076923076923, 'number': 1332}, 'eval_overall_precision': 0.661106590724166, 'eval_overall_recall': 0.6427158866183257, 'eval_overall_f1': 0.6517815361989439, 'e

20

In [37]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [38]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [39]:
results = {}
trainer_all = {}
s = splits[2]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: semantic


100%|██████████| 10392/10392 [00:00<00:00, 1229020.36it/s]
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([13]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([13, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2185/2185 [00:00<00:00, 16481.38 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_15840\2617588520.py:175: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Jurisprudencia,Legislacao,Local,Organizacao,Pessoa,Tempo,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.091279,"{'precision': 0.6372549019607843, 'recall': 0.8125, 'f1': 0.7142857142857142, 'number': 160}","{'precision': 0.8098859315589354, 'recall': 0.8987341772151899, 'f1': 0.8520000000000001, 'number': 237}","{'precision': 0.646551724137931, 'recall': 0.49019607843137253, 'f1': 0.5576208178438662, 'number': 153}","{'precision': 0.7180043383947939, 'recall': 0.6377649325626205, 'f1': 0.6755102040816328, 'number': 519}","{'precision': 0.8629737609329446, 'recall': 0.9381933438985737, 'f1': 0.8990129081245255, 'number': 631}","{'precision': 0.9943820224719101, 'recall': 0.9315789473684211, 'f1': 0.9619565217391305, 'number': 190}",0.795597,0.803175,0.799368,0.975074,0.975074,0.866756,0.974284
2,0.116400,0.071242,"{'precision': 0.6903553299492385, 'recall': 0.85, 'f1': 0.7619047619047619, 'number': 160}","{'precision': 0.7971014492753623, 'recall': 0.9282700421940928, 'f1': 0.8576998050682261, 'number': 237}","{'precision': 0.7024793388429752, 'recall': 0.5555555555555556, 'f1': 0.6204379562043797, 'number': 153}","{'precision': 0.7579505300353356, 'recall': 0.8265895953757225, 'f1': 0.7907834101382488, 'number': 519}","{'precision': 0.8885400313971743, 'recall': 0.8969889064976229, 'f1': 0.8927444794952683, 'number': 631}","{'precision': 0.9789473684210527, 'recall': 0.9789473684210527, 'f1': 0.9789473684210527, 'number': 190}",0.816306,0.858201,0.836729,0.980962,0.980962,0.893595,0.980640
3,0.030700,0.075952,"{'precision': 0.7417582417582418, 'recall': 0.84375, 'f1': 0.7894736842105263, 'number': 160}","{'precision': 0.8287937743190662, 'recall': 0.8987341772151899, 'f1': 0.8623481781376519, 'number': 237}","{'precision': 0.7857142857142857, 'recall': 0.6470588235294118, 'f1': 0.7096774193548386, 'number': 153}","{'precision': 0.8280922431865828, 'recall': 0.7610789980732178, 'f1': 0.7931726907630523, 'number': 519}","{'precision': 0.92, 'recall': 0.9112519809825673, 'f1': 0.9156050955414013, 'number': 631}","{'precision': 0.9944444444444445, 'recall': 0.9421052631578948, 'f1': 0.9675675675675677, 'number': 190}",0.864104,0.844444,0.854161,0.981551,0.981551,0.902618,0.981224
4,0.014000,0.076886,"{'precision': 0.7555555555555555, 'recall': 0.85, 'f1': 0.7999999999999998, 'number': 160}","{'precision': 0.8700787401574803, 'recall': 0.9324894514767933, 'f1': 0.90020366598778, 'number': 237}","{'precision': 0.7022900763358778, 'recall': 0.6013071895424836, 'f1': 0.647887323943662, 'number': 153}","{'precision': 0.781981981981982, 'recall': 0.8362235067437379, 'f1': 0.808193668528864, 'number': 519}","{'precision': 0.9197431781701445, 'recall': 0.9080824088748018, 'f1': 0.9138755980861244, 'number': 631}","{'precision': 0.994475138121547, 'recall': 0.9473684210526315, 'f1': 0.9703504043126685, 'number': 190}",0.850312,0.865608,0.857892,0.983145,0.983145,0.904250,0.983099
5,0.008200,0.072553,"{'precision': 0.7666666666666667, 'recall': 0.8625, 'f1': 0.8117647058823529, 'number': 160}","{'precision': 0.872, 'recall': 0.919831223628692, 'f1': 0.8952772073921971, 'number': 237}","{'precision': 0.7580645161290323, 'recall': 0.6143790849673203, 'f1': 0.6787003610108303, 'number': 153}","{'precision': 0.8326923076923077, 'recall': 0.8342967244701349, 'f1': 0.8334937439846006, 'number': 519}","{'precision': 0.9322834645669291, 'recall': 0.9381933438985737, 'f1': 0.9352290679304898, 'number': 631}","{'precision': 0.9945054945054945, 'recall': 0.9526315789473684, 'f1': 0.9731182795698924, 'number': 190}",0.875727,0.876190,0.875959,0.984715,0.984715,0.912854,0.984495


F1 Macro: 0.8676006522213805
F1 Micro: 0.9781740540113084
F1 Weighted: 0.9780081304928313
{'eval_loss': 0.08109546452760696, 'eval_JURISPRUDENCIA': {'precision': 0.8028169014084507, 'recall': 0.7916666666666666, 'f1': 0.7972027972027971, 'number': 144}, 'eval_LEGISLACAO': {'precision': 0.9065934065934066, 'recall': 0.9510086455331412, 'f1': 0.9282700421940928, 'number': 347}, 'eval_LOCAL': {'precision': 0.8867924528301887, 'recall': 0.8867924528301887, 'f1': 0.8867924528301887, 'number': 53}, 'eval_ORGANIZACAO': {'precision': 0.8556149732620321, 'recall': 0.9169054441260746, 'f1': 0.8852005532503459, 'number': 349}, 'eval_PESSOA': {'precision': 0.6984126984126984, 'recall': 0.946236559139785, 'f1': 0.8036529680365297, 'number': 93}, 'eval_TEMPO': {'precision': 0.979381443298969, 'recall': 0.95, 'f1': 0.9644670050761421, 'number': 200}, 'eval_overall_precision': 0.8691141260973663, 'eval_overall_recall': 0.918212478920742, 'eval_overall_f1': 0.8929889298892989, 'eval_overall_accuracy': 

20

In [40]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [41]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [42]:
results = {}
trainer_all = {}
s = splits[3]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_len


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([13]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([13, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2078/2078 [00:00<00:00, 7999.39 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_15840\2617588520.py:175: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Jurisprudencia,Legislacao,Local,Organizacao,Pessoa,Tempo,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.054669,"{'precision': 0.7018633540372671, 'recall': 0.889763779527559, 'f1': 0.7847222222222221, 'number': 127}","{'precision': 0.8608058608058609, 'recall': 0.9251968503937008, 'f1': 0.8918406072106262, 'number': 254}","{'precision': 0.8709677419354839, 'recall': 0.84375, 'f1': 0.8571428571428571, 'number': 64}","{'precision': 0.7801608579088471, 'recall': 0.8558823529411764, 'f1': 0.8162692847124825, 'number': 340}","{'precision': 0.8465116279069768, 'recall': 0.8708133971291866, 'f1': 0.8584905660377359, 'number': 209}","{'precision': 0.9627329192546584, 'recall': 0.950920245398773, 'f1': 0.9567901234567902, 'number': 163}",0.827309,0.890233,0.857619,0.984808,0.984808,0.923399,0.985025
2,0.129100,0.044147,"{'precision': 0.7891156462585034, 'recall': 0.9133858267716536, 'f1': 0.8467153284671532, 'number': 127}","{'precision': 0.9038461538461539, 'recall': 0.9251968503937008, 'f1': 0.9143968871595332, 'number': 254}","{'precision': 0.8309859154929577, 'recall': 0.921875, 'f1': 0.874074074074074, 'number': 64}","{'precision': 0.8717948717948718, 'recall': 0.9, 'f1': 0.8856729377713459, 'number': 340}","{'precision': 0.916256157635468, 'recall': 0.8899521531100478, 'f1': 0.9029126213592233, 'number': 209}","{'precision': 0.9629629629629629, 'recall': 0.9570552147239264, 'f1': 0.9599999999999999, 'number': 163}",0.886097,0.914434,0.900043,0.987708,0.987708,0.944813,0.987770
3,0.030800,0.048298,"{'precision': 0.8208955223880597, 'recall': 0.8661417322834646, 'f1': 0.842911877394636, 'number': 127}","{'precision': 0.930327868852459, 'recall': 0.8937007874015748, 'f1': 0.9116465863453815, 'number': 254}","{'precision': 0.8955223880597015, 'recall': 0.9375, 'f1': 0.9160305343511451, 'number': 64}","{'precision': 0.9009009009009009, 'recall': 0.8823529411764706, 'f1': 0.8915304606240713, 'number': 340}","{'precision': 0.915, 'recall': 0.8755980861244019, 'f1': 0.8948655256723717, 'number': 209}","{'precision': 0.968944099378882, 'recall': 0.9570552147239264, 'f1': 0.962962962962963, 'number': 163}",0.909570,0.895419,0.902439,0.988993,0.988993,0.948855,0.988897
4,0.015000,0.041125,"{'precision': 0.8085106382978723, 'recall': 0.8976377952755905, 'f1': 0.8507462686567165, 'number': 127}","{'precision': 0.92578125, 'recall': 0.9330708661417323, 'f1': 0.9294117647058823, 'number': 254}","{'precision': 0.9090909090909091, 'recall': 0.9375, 'f1': 0.923076923076923, 'number': 64}","{'precision': 0.9075144508670521, 'recall': 0.9235294117647059, 'f1': 0.9154518950437318, 'number': 340}","{'precision': 0.9215686274509803, 'recall': 0.8995215311004785, 'f1': 0.9104116222760289, 'number': 209}","{'precision': 0.9634146341463414, 'recall': 0.9693251533742331, 'f1': 0.966360856269113, 'number': 163}",0.909941,0.925670,0.917738,0.990476,0.990476,0.954418,0.990461
5,0.007700,0.042805,"{'precision': 0.8405797101449275, 'recall': 0.9133858267716536, 'f1': 0.8754716981132076, 'number': 127}","{'precision': 0.93359375, 'recall': 0.9409448818897638, 'f1': 0.9372549019607842, 'number': 254}","{'precision': 0.9375, 'recall': 0.9375, 'f1': 0.9375, 'number': 64}","{'precision': 0.8985507246376812, 'recall': 0.9117647058823529, 'f1': 0.9051094890510949, 'number': 340}","{'precision': 0.9158415841584159, 'recall': 0.8851674641148325, 'f1': 0.9002433090024331, 'number': 209}","{'precision': 0.9575757575757575, 'recall': 0.9693251533742331, 'f1': 0.9634146341463414, 'number': 163}",0.912821,0.923077,0.917920,0.990443,0.990443,0.953870,0.990434


F1 Macro: 0.9433885533248542
F1 Micro: 0.9892717867091126
F1 Weighted: 0.989286709570346
{'eval_loss': 0.05115306004881859, 'eval_JURISPRUDENCIA': {'precision': 0.8121387283236994, 'recall': 0.8753894080996885, 'f1': 0.8425787106446776, 'number': 321}, 'eval_LEGISLACAO': {'precision': 0.9137055837563451, 'recall': 0.9507042253521126, 'f1': 0.9318377911993097, 'number': 568}, 'eval_LOCAL': {'precision': 0.7874396135265701, 'recall': 0.8358974358974359, 'f1': 0.8109452736318409, 'number': 195}, 'eval_ORGANIZACAO': {'precision': 0.8764783180026281, 'recall': 0.892904953145917, 'f1': 0.8846153846153847, 'number': 747}, 'eval_PESSOA': {'precision': 0.9386363636363636, 'recall': 0.9560185185185185, 'f1': 0.9472477064220183, 'number': 432}, 'eval_TEMPO': {'precision': 0.9671232876712329, 'recall': 0.9592391304347826, 'f1': 0.9631650750341064, 'number': 368}, 'eval_overall_precision': 0.8918819188191882, 'eval_overall_recall': 0.9186621056632459, 'eval_overall_f1': 0.9050739561879799, 'eval_ov

20

In [43]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [44]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch



time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [45]:
results = {}
trainer_all = {}
s = splits[4]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_rare


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([13]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([13, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2078/2078 [00:00<00:00, 6049.18 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_15840\2617588520.py:175: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Jurisprudencia,Legislacao,Local,Organizacao,Pessoa,Tempo,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.058134,"{'precision': 0.6894409937888198, 'recall': 0.8409090909090909, 'f1': 0.7576791808873721, 'number': 132}","{'precision': 0.8841059602649006, 'recall': 0.9468085106382979, 'f1': 0.9143835616438356, 'number': 282}","{'precision': 0.7294117647058823, 'recall': 0.8378378378378378, 'f1': 0.779874213836478, 'number': 74}","{'precision': 0.7784256559766763, 'recall': 0.8291925465838509, 'f1': 0.8030075187969925, 'number': 322}","{'precision': 0.9563106796116505, 'recall': 0.9656862745098039, 'f1': 0.9609756097560976, 'number': 204}","{'precision': 0.9602649006622517, 'recall': 0.9354838709677419, 'f1': 0.9477124183006537, 'number': 155}",0.840545,0.897348,0.868018,0.984303,0.984303,0.928285,0.984459
2,0.123100,0.050733,"{'precision': 0.8705035971223022, 'recall': 0.9166666666666666, 'f1': 0.8929889298892989, 'number': 132}","{'precision': 0.8959731543624161, 'recall': 0.9468085106382979, 'f1': 0.9206896551724137, 'number': 282}","{'precision': 0.8831168831168831, 'recall': 0.918918918918919, 'f1': 0.9006622516556292, 'number': 74}","{'precision': 0.8729641693811075, 'recall': 0.8322981366459627, 'f1': 0.8521462639109699, 'number': 322}","{'precision': 0.9563106796116505, 'recall': 0.9656862745098039, 'f1': 0.9609756097560976, 'number': 204}","{'precision': 0.974025974025974, 'recall': 0.967741935483871, 'f1': 0.970873786407767, 'number': 155}",0.906859,0.916168,0.911489,0.986751,0.986751,0.946339,0.986619
3,0.029000,0.052647,"{'precision': 0.9375, 'recall': 0.9090909090909091, 'f1': 0.923076923076923, 'number': 132}","{'precision': 0.900990099009901, 'recall': 0.9680851063829787, 'f1': 0.9333333333333335, 'number': 282}","{'precision': 0.8985507246376812, 'recall': 0.8378378378378378, 'f1': 0.8671328671328672, 'number': 74}","{'precision': 0.8757575757575757, 'recall': 0.8975155279503105, 'f1': 0.8865030674846625, 'number': 322}","{'precision': 0.9655172413793104, 'recall': 0.9607843137254902, 'f1': 0.9631449631449631, 'number': 204}","{'precision': 0.9806451612903225, 'recall': 0.9806451612903225, 'f1': 0.9806451612903225, 'number': 155}",0.919192,0.934132,0.926602,0.989233,0.989233,0.953318,0.989233
4,0.013100,0.046552,"{'precision': 0.9044117647058824, 'recall': 0.9318181818181818, 'f1': 0.9179104477611939, 'number': 132}","{'precision': 0.911864406779661, 'recall': 0.9539007092198581, 'f1': 0.9324090121317158, 'number': 282}","{'precision': 0.8918918918918919, 'recall': 0.8918918918918919, 'f1': 0.8918918918918919, 'number': 74}","{'precision': 0.8882175226586103, 'recall': 0.9130434782608695, 'f1': 0.9004594180704442, 'number': 322}","{'precision': 0.9658536585365853, 'recall': 0.9705882352941176, 'f1': 0.9682151589242053, 'number': 204}","{'precision': 0.9807692307692307, 'recall': 0.9870967741935484, 'f1': 0.9839228295819936, 'number': 155}",0.921470,0.943541,0.932375,0.989303,0.989303,0.957520,0.989312
5,0.008600,0.049623,"{'precision': 0.8985507246376812, 'recall': 0.9393939393939394, 'f1': 0.9185185185185185, 'number': 132}","{'precision': 0.9121621621621622, 'recall': 0.9574468085106383, 'f1': 0.9342560553633218, 'number': 282}","{'precision': 0.868421052631579, 'recall': 0.8918918918918919, 'f1': 0.88, 'number': 74}","{'precision': 0.9012345679012346, 'recall': 0.906832298136646, 'f1': 0.9040247678018576, 'number': 322}","{'precision': 0.9607843137254902, 'recall': 0.9607843137254902, 'f1': 0.9607843137254902, 'number': 204}","{'precision': 0.9743589743589743, 'recall': 0.9806451612903225, 'f1': 0.977491961414791, 'number': 155}",0.921273,0.940975,0.931020,0.989722,0.989722,0.960228,0.989741


F1 Macro: 0.9298025583281734
F1 Micro: 0.9860707618553171
F1 Weighted: 0.9860347646932086
{'eval_loss': 0.06729217618703842, 'eval_JURISPRUDENCIA': {'precision': 0.8135593220338984, 'recall': 0.9094736842105263, 'f1': 0.8588469184890656, 'number': 475}, 'eval_LEGISLACAO': {'precision': 0.8397711015736766, 'recall': 0.9003067484662577, 'f1': 0.8689859363434493, 'number': 652}, 'eval_LOCAL': {'precision': 0.7811158798283262, 'recall': 0.8387096774193549, 'f1': 0.8088888888888889, 'number': 217}, 'eval_ORGANIZACAO': {'precision': 0.8305252725470763, 'recall': 0.8568507157464212, 'f1': 0.8434826371414192, 'number': 978}, 'eval_PESSOA': {'precision': 0.9242424242424242, 'recall': 0.9119601328903655, 'f1': 0.9180602006688963, 'number': 602}, 'eval_TEMPO': {'precision': 0.9581818181818181, 'recall': 0.9547101449275363, 'f1': 0.956442831215971, 'number': 552}, 'eval_overall_precision': 0.8614491150442478, 'eval_overall_recall': 0.89614499424626, 'eval_overall_f1': 0.8784545967287084, 'eval_ove

20

In [46]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [47]:
#del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


#time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [48]:
lener_full

Dataset({
    features: ['id', 'tokens', 'ner_tags', 'ner_tags_str'],
    num_rows: 10392
})

In [ ]:
results = {}
trainer_all = {}
s = splits[5]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: std


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([13]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([13, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\user\AppData\Local\Temp\ipykernel_15840\2617588520.py:175: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Jurisprudencia,Legislacao,Local,Organizacao,Pessoa,Tempo,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.125802,"{'precision': 0.5930232558139535, 'recall': 0.4927536231884058, 'f1': 0.5382585751978891, 'number': 207}","{'precision': 0.7444933920704846, 'recall': 0.8513853904282116, 'f1': 0.7943595769682726, 'number': 397}","{'precision': 0.5, 'recall': 0.3761467889908257, 'f1': 0.42931937172774876, 'number': 109}","{'precision': 0.640387275242047, 'recall': 0.8253119429590018, 'f1': 0.7211838006230529, 'number': 561}","{'precision': 0.7410468319559229, 'recall': 0.867741935483871, 'f1': 0.799405646359584, 'number': 310}","{'precision': 0.9055793991416309, 'recall': 0.9017094017094017, 'f1': 0.9036402569593148, 'number': 234}",0.702516,0.783278,0.740702,0.964801,0.964801,0.807854,0.963289
2,0.124600,0.128795,"{'precision': 0.5207373271889401, 'recall': 0.5458937198067633, 'f1': 0.5330188679245282, 'number': 207}","{'precision': 0.6, 'recall': 0.8690176322418136, 'f1': 0.7098765432098766, 'number': 397}","{'precision': 0.3881278538812785, 'recall': 0.7798165137614679, 'f1': 0.5182926829268293, 'number': 109}","{'precision': 0.7016574585635359, 'recall': 0.679144385026738, 'f1': 0.6902173913043478, 'number': 561}","{'precision': 0.7623456790123457, 'recall': 0.7967741935483871, 'f1': 0.779179810725552, 'number': 310}","{'precision': 0.899581589958159, 'recall': 0.9188034188034188, 'f1': 0.909090909090909, 'number': 234}",0.654700,0.762376,0.704447,0.960574,0.960574,0.816001,0.961016
3,0.030300,0.112543,"{'precision': 0.5927601809954751, 'recall': 0.6328502415458938, 'f1': 0.6121495327102804, 'number': 207}","{'precision': 0.6705426356589147, 'recall': 0.871536523929471, 'f1': 0.7579408543263965, 'number': 397}","{'precision': 0.524390243902439, 'recall': 0.7889908256880734, 'f1': 0.63003663003663, 'number': 109}","{'precision': 0.7370304114490162, 'recall': 0.7344028520499108, 'f1': 0.7357142857142858, 'number': 561}","{'precision': 0.7904761904761904, 'recall': 0.8032258064516129, 'f1': 0.7968000000000001, 'number': 310}","{'precision': 0.8934426229508197, 'recall': 0.9316239316239316, 'f1': 0.9121338912133892, 'number': 234}",0.714215,0.793179,0.751629,0.968056,0.968056,0.855807,0.968010


In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

Acesso negado - .
Arquivo n�o encontrado  - -NAME
Arquivo n�o encontrado  - -EXEC
Arquivo n�o encontrado  - RM
Arquivo n�o encontrado  - -RF
Arquivo n�o encontrado  - {}
Arquivo n�o encontrado  - +


In [ ]:
results = {}
trainer_all = {}
s = splits[6]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(lener_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: advs
Selecionando 2078 sentenças para teste…
  0 selecionadas…
  25 selecionadas…
  50 selecionadas…
  74 selecionadas…
  95 selecionadas…
  118 selecionadas…
  135 selecionadas…
  158 selecionadas…
  180 selecionadas…
  201 selecionadas…
  222 selecionadas…
  243 selecionadas…
  265 selecionadas…
  288 selecionadas…
  310 selecionadas…
  335 selecionadas…
  360 selecionadas…
  381 selecionadas…
  399 selecionadas…
  419 selecionadas…
  440 selecionadas…
  458 selecionadas…
  481 selecionadas…
  500 selecionadas…
  520 selecionadas…
  536 selecionadas…
  550 selecionadas…
  563 selecionadas…
  584 selecionadas…
  605 selecionadas…
  617 selecionadas…
  636 selecionadas…
  654 selecionadas…
  669 selecionadas…
  681 selecionadas…
  704 selecionadas…
  727 selecionadas…
  742 selecionadas…
  760 selecionadas…
  776 selecionadas…
  792 selecionadas…
  814 selecionadas…
  828 selecionadas…
  846 selecionadas…
  867 selecionadas…
  883 selecionadas…
  902 selecionadas…


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([13]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([13, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 1039/1039 [00:00<00:00, 8032.00 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_21248\2617588520.py:175: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Jurisprudencia,Legislacao,Local,Organizacao,Pessoa,Tempo,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.052417,"{'precision': 0.7793103448275862, 'recall': 0.8014184397163121, 'f1': 0.7902097902097902, 'number': 141}","{'precision': 0.8287671232876712, 'recall': 0.9201520912547528, 'f1': 0.8720720720720719, 'number': 263}","{'precision': 0.803030303030303, 'recall': 0.8833333333333333, 'f1': 0.8412698412698413, 'number': 60}","{'precision': 0.7823834196891192, 'recall': 0.8228882833787466, 'f1': 0.802124833997344, 'number': 367}","{'precision': 0.927461139896373, 'recall': 0.9572192513368984, 'f1': 0.9421052631578948, 'number': 187}","{'precision': 0.9640718562874252, 'recall': 0.9640718562874252, 'f1': 0.9640718562874252, 'number': 167}",0.840673,0.886076,0.862777,0.985184,0.985184,0.929283,0.985219
2,0.127400,0.041580,"{'precision': 0.7972972972972973, 'recall': 0.8368794326241135, 'f1': 0.8166089965397924, 'number': 141}","{'precision': 0.9118773946360154, 'recall': 0.9049429657794676, 'f1': 0.9083969465648855, 'number': 263}","{'precision': 0.8548387096774194, 'recall': 0.8833333333333333, 'f1': 0.8688524590163934, 'number': 60}","{'precision': 0.8492063492063492, 'recall': 0.8746594005449592, 'f1': 0.861744966442953, 'number': 367}","{'precision': 0.9473684210526315, 'recall': 0.9625668449197861, 'f1': 0.9549071618037135, 'number': 187}","{'precision': 0.9704142011834319, 'recall': 0.9820359281437125, 'f1': 0.9761904761904762, 'number': 167}",0.889073,0.906329,0.897618,0.989966,0.989966,0.952309,0.990023
3,0.030500,0.040319,"{'precision': 0.85, 'recall': 0.8439716312056738, 'f1': 0.8469750889679715, 'number': 141}","{'precision': 0.8981818181818182, 'recall': 0.9391634980988594, 'f1': 0.9182156133828997, 'number': 263}","{'precision': 0.9285714285714286, 'recall': 0.8666666666666667, 'f1': 0.896551724137931, 'number': 60}","{'precision': 0.8451776649746193, 'recall': 0.9073569482288828, 'f1': 0.8751642575558477, 'number': 367}","{'precision': 0.9574468085106383, 'recall': 0.9625668449197861, 'f1': 0.96, 'number': 187}","{'precision': 0.9649122807017544, 'recall': 0.9880239520958084, 'f1': 0.9763313609467456, 'number': 167}",0.895425,0.924895,0.909921,0.991212,0.991212,0.957695,0.991218
4,0.014900,0.042443,"{'precision': 0.8843537414965986, 'recall': 0.9219858156028369, 'f1': 0.9027777777777778, 'number': 141}","{'precision': 0.8880866425992779, 'recall': 0.935361216730038, 'f1': 0.9111111111111111, 'number': 263}","{'precision': 0.9298245614035088, 'recall': 0.8833333333333333, 'f1': 0.905982905982906, 'number': 60}","{'precision': 0.8870967741935484, 'recall': 0.8991825613079019, 'f1': 0.8930987821380244, 'number': 367}","{'precision': 0.9521276595744681, 'recall': 0.9572192513368984, 'f1': 0.9546666666666667, 'number': 187}","{'precision': 0.9760479041916168, 'recall': 0.9760479041916168, 'f1': 0.9760479041916168, 'number': 167}",0.911424,0.929114,0.920184,0.991009,0.991009,0.956755,0.991039
5,0.008100,0.043064,"{'precision': 0.8827586206896552, 'recall': 0.9078014184397163, 'f1': 0.8951048951048951, 'number': 141}","{'precision': 0.9051094890510949, 'recall': 0.9429657794676806, 'f1': 0.9236499068901304, 'number': 263}","{'precision': 0.8870967741935484, 'recall': 0.9166666666666666, 'f1': 0.9016393442622951, 'number': 60}","{'precision': 0.8756613756613757, 'recall': 0.9019073569482289, 'f1': 0.8885906040268456, 'number': 367}","{'precision': 0.9523809523809523, 'recall': 0.9625668449197861, 'f1': 0.9574468085106382, 'number': 187}","{'precision': 0.9763313609467456, 'recall': 0.9880239520958084, 'f1': 0.9821428571428571, 'number': 167}",0.909614,0.934177,0.921732,0.991515,0.991515,0.959301,0.991541


F1 Macro: 0.9191097151935363
F1 Micro: 0.9857580004003775
F1 Weighted: 0.985710037273226
{'eval_loss': 0.06945380568504333, 'eval_JURISPRUDENCIA': {'precision': 0.7518796992481203, 'recall': 0.821917808219178, 'f1': 0.7853403141361257, 'number': 365}, 'eval_LEGISLACAO': {'precision': 0.8629579375848032, 'recall': 0.9284671532846716, 'f1': 0.8945147679324894, 'number': 685}, 'eval_LOCAL': {'precision': 0.7857142857142857, 'recall': 0.8608695652173913, 'f1': 0.8215767634854773, 'number': 115}, 'eval_ORGANIZACAO': {'precision': 0.8389694041867954, 'recall': 0.8389694041867954, 'f1': 0.8389694041867956, 'number': 621}, 'eval_PESSOA': {'precision': 0.9108108108108108, 'recall': 0.9387186629526463, 'f1': 0.9245541838134431, 'number': 359}, 'eval_TEMPO': {'precision': 0.9924812030075187, 'recall': 0.9295774647887324, 'f1': 0.96, 'number': 284}, 'eval_overall_precision': 0.8562921794362842, 'eval_overall_recall': 0.8880197612186085, 'eval_overall_f1': 0.8718674211802748, 'eval_overall_accuracy

20

In [ ]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [ ]:
df = ner_collector.to_dataframes()

In [ ]:
df

(       split  eval_loss  overall_precision  overall_recall  overall_f1  \
 0        loc   0.071585           0.784648        0.817778    0.800871   
 1    reverse   0.618983           0.661107        0.642716    0.651782   
 2   semantic   0.081094           0.869114        0.918212    0.892989   
 3   heur_len   0.051157           0.891882        0.918662    0.905074   
 4  heur_rare   0.067291           0.861449        0.896145    0.878455   
 5        std   0.075426           0.829962        0.864384    0.846824   
 6       advs   0.069454           0.856292        0.888020    0.871867   
 
    overall_accuracy  f1_micro  f1_macro  f1_weighted  runtime_s  \
 0          0.987900  0.987900  0.874908     0.987743     6.4417   
 1          0.905898  0.905898  0.721392     0.897010    32.5871   
 2          0.978174  0.978174  0.867601     0.978008     8.2151   
 3          0.989272  0.989272  0.943389     0.989287    20.1880   
 4          0.986071  0.986071  0.929803     0.986035    2

In [ ]:
nome_modelo = MODEL_NAME.split("/")[1]

In [ ]:
ner_collector.to_csv(base_name=nome_modelo)

('albertina-base_overall_metrics.csv', 'albertina-base_label_metrics.csv')